In [1]:
import warnings
warnings.filterwarnings('ignore')

# LangChain의 LCEL(LangChain Expression Language)

LangChain의 컴포넌트를 조합하여 복잡한 작업 흐름을 쉽게 구성할 수 있게 해준다.

# 기본 설정

## .env 환경 변수

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 기본 라이브러리

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

##  벡터 저장소 로드

In [4]:
# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# 벡터 저장소 설정
# Chroma() 클래스로 로컬에 저장할 때 사용한 임베딩 모델, 테이블 이름, 테이블이 저장된 경로를 넘겨서 벡터 저장소의 문서들을 얻어온다.
vectorstore = Chroma(
    embedding_function=embeddings, # 로컬에 저장할 때 텍스트를 벡터로 변환할 때 사용한 OpenAIEmbeddings로 만든 임베딩 모델을 지정한다.
    collection_name='chroma_test', # 읽어올 Chroma 데이터베이스 내부의 테이블(벡터 저장소) 이름을 지정한다.
    persist_directory='./chroma_db', # 벡터 저장소가 물리적인 파일로 저장된 디렉토리(폴더)를 지정한다.
)

print(f'벡터 저장소에 저장된 문서 개수: {vectorstore._collection.count()}')

벡터 저장소에 저장된 문서 개수: 5


# Prompt와 LLM 연결하기

LCEL을 사용해서 프롬프트와 LLM 연결

<img src="./lcel1.png" width="1200" align="left" />

동적 프롬프트 템플릿 생성 및 활용

<img src="./lcel2.png" width="600" align="left" />

LangChain을 사용해서 LLM(대화형 AI)에게 전달할 프롬프트를 설계한다.  
단순히 질문만 던지는 것이 아니라, AI의 역할(Persona, 정체성)과 사용자의 질문 형식을 미리 정의하는 틀을 만든다.

In [5]:
# 언어 모델을 만든다.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_completion_tokens=150)

# 프롬프트를 대화 메시지를 큰 리스트에 역할과 메시지를 튜플 형태로 정의한다.
# 역할은 'human', 'user', 'ai', 'assistant', 'system' 중 1개를 사용해야 한다.
messages = [
    # 시스템 메시지라고 하며, AI에게 '너는 어떤 존재인가?'라는 역할을 부여한다.
    ('system', '당신은 유능한 나만의 인공지능 비서입니다.'), # SystemMessagePromptTemplate
    # 사용자 메시지라고 하며, {query}는 나중에 실제 질문 내용으로 치환될 변수(placeholder) 자리를 의미한다.
    ('human', '{query}'), # HumanMessagePromptTemplate
]

# 프롬프트 템플릿을 생성한다.
# 위에서 정의한 튜플이 저장된 리스트 형태의 메시지 구조를 바탕으로, LangChain이 인식할 수 있는 템플릿 객체로 변환한다.
# 이 프롬프트 객체에 질문만 넣어주면 모델이 이해할 수 있는 복잡한 메시지 구조를 만들어 준다.
# from_messages() 메소드에 프롬프트로 구성할 내용을 넘겨서 프롬프트를 만든다.
prompt = ChatPromptTemplate.from_messages(messages)
prompt

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 유능한 나만의 인공지능 비서입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

프롬프트 내부의 변수 목록 확인하기

In [6]:
print(prompt.input_variables)

['query']


프롬프트 내부의 변수에 실제 내용을 채워넣어, 모델에 전달할 최종 메시지 형태 만들기, 변수에 실제 내용을 넣어서 프롬프트 완성하기

In [7]:
# format() 메소드의 인수로 '변수이름=메시지' 형태로 실제 내용을 넣어준다.
prompt_text = prompt.format(query='테슬라 창업자는 누구인가요?')
print(prompt_text)

System: 당신은 유능한 나만의 인공지능 비서입니다.
Human: 테슬라 창업자는 누구인가요?


프롬프트를 AI 모델에게 실제로 보내서 답변을 받아내는 RAG 프로세스를 실행한다.

In [8]:
# LLM 호출 및 답변을 생성한다.
# LCEL 문법을 사용하지(파이프라인을 연결하지) 않고 RAG를 실행할 때는 프롬프트에 format() 메소드로 변수에 실제 내용을 채워넣고 전달한다.
response = llm.invoke(prompt_text)

# RAG 프로세스의 실행 결과는 AIMessage 객체이고 여기에는 답변 내용뿐만 아니라 사용된 토큰 수, 수행 시간 등 다양한 메타 데이터가 포함되어 있다.
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 이후 엘론 머스크가 2004년에 투자자로 참여하면서 CEO로 취임하게 되었고, 회사의 비전과 방향성을 주도하게 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 36, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_53c3b1e564', 'id': 'chatcmpl-EKC6rA8nbExkyUo7x8HcilMYJSxk0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a069d2-c501-79b2-b616-02ee3643fb75-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_t

RAG 프로세스 실행 결과에서 답변 내용만 추출한다.

In [9]:
print(response.content)

테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 이후 엘론 머스크가 2004년에 투자자로 참여하면서 CEO로 취임하게 되었고, 회사의 비전과 방향성을 주도하게 되었습니다.


## LangChain의 LCEL을 사용해서 프롬프트와 모델 연결해서 chain 만들기

`|(파이프 연산자)`는 `|` 왼쪽의 출력을 `|` 오른쪽의 입력으로 보낸다.

In [10]:
# 체인을 생성한다. 파이프라인을 연결한다.
# prompt가 사용자의 입력을 받아 메시지 형태로 가공한다. => 가공된 메시지를 모델에게 전달한다.
# chain에 prompt와 llm이 합쳐진 새로운 객체가 생성된다. 이전 처럼 프롬프트를 만들고 모델을 따로 호출할 필요 없이 chain만 실행하면 답변까지 한 번에 나온다.
chain = prompt | llm
chain

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 유능한 나만의 인공지능 비서입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature

체인의 input_schema 속성은 LangChain의 모든 컴포넌트(Runnable)의 입력 형식을 정의한 스키마(데이터 구조) 객체를 얻어온다.  
스키마에서 schema() 메소드를 실행하면 이 객체를 JSON 형식의 딕셔너리로 변환한다.  

In [11]:
print(chain.input_schema.schema())

{'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'PromptInput', 'type': 'object'}


파이썬의 데이터를 `보기 좋게 출력(Pretty Print)`하기 위해 pprint를 import 한다.

In [12]:
from pprint import pprint

일반 print() 함수는 딕셔너리나 리스트 등 복잡한 구성을 가지는 데이터를 출력할 때 한 줄로 길게 늘어뜨려 읽기 힘들게 보여주지만, pprint() 함수는 이를 줄 바꿈과 들여쓰기를 적용해 깔끔하게 정돈해서 출력한다.

In [13]:
pprint(chain.input_schema.schema())

{'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'PromptInput',
 'type': 'object'}


## chain 실행하기

앞에서 `chain = prompt | llm` 형태로 연결해둔 체인을 실제로 동작시켜 질문을 던지고, AI의 최종 답변을 받아온다.

LCEL 문법을 사용하지 않을 경우 format() 메소드로 프롬프트에 실제 내용을 채워넣고 실행하지만 LCEL 문법을 사용하는 경우 프롬프트에 채워넣을 변수 이름을 key로 하고 메시지를 value로 하는 딕셔너리를 넘겨서 실행한다.

In [14]:
# invoke() 메소드의 인수는 {변수 이름: 메시지} 형태의 딕셔너리를 넘겨야 한다.
# invoke() 메소드가 실행되면 인수로 지정된 '테슬라 창업자는 누구인가요?' 메시지가 프롬프트의 {query}라는 변수로 전달되서 프롬프트가 완성되고 완성된 프롬프트 
# 내용이 AI 모델의 입력으로 전달된다.
# AI 모델은 입력받은 프롬프트를 LLM에게 던지고 답변 내용과 메타 데이터가 저장된 AIMessage 객체 형태의 응답을 받는다.
response = chain.invoke({'query': '테슬라 창업자는 누구인가요?'})
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 성장에 큰 영향을 미쳤습니다. 이후 그는 테슬라의 가장 유명한 얼굴이 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 37, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da56f7d23d', 'id': 'chatcmpl-EKCBTXt1LwbLnzGfs3dxGp6KEqjOZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a069d7-232e-7b01-8f5c-79cbd3619c54-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

In [15]:
print(response.content)

테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 성장에 큰 영향을 미쳤습니다. 이후 그는 테슬라의 가장 유명한 얼굴이 되었습니다.


In [16]:
# chain이 실행하는 프롬프트는 변수가 1개인 프롬프트는 LangChain이 알아서 {query} 변수에 자동으로 내용을 채워준다.
response = chain.invoke('테슬라 창업자는 누구인가요?')
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 성장에 큰 영향을 미쳤습니다. 이후 그는 테슬라의 가장 잘 알려진 얼굴이 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 37, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da56f7d23d', 'id': 'chatcmpl-EKCBfl6SApjrUnpArRRclZgYpYBch', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a069d7-5052-7090-9c1a-62fff61f271a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_t

# Prompt와 LLM과 OutputParser 연결하기

## 문자열 파싱 - StrOutputParser

모델의 응답인 AIMessage 객체에서 순수 응답 결과(content)를 문자열만 얻어오기 위해서 StrOutputParser를 import 한다.

In [17]:
from langchain_core.output_parsers import StrOutputParser

In [18]:
# 문자열 파서 StrOutputParser의 객체를 만든다. AIMessage 객체에서 content 부분만 얻어올 수 있다.
output_parser = StrOutputParser()

# 문자열 파서를 실행한다.
# invoke() 메소드의 인수로 LLM의 응답 결과(AIMessage 객체)를 넘겨셔 문자열만 얻어온다. 인수로 넘긴 내용이 순수한 문자열이면 문자열이 그대로 반환된다.
output_parser.invoke(response)

'테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 성장에 큰 영향을 미쳤습니다. 이후 그는 테슬라의 가장 잘 알려진 얼굴이 되었습니다.'

문자열 파서를 LCEL 방식으로 연결한다.

In [19]:
# invoke() 메소드가 실행되면 질문이 프롬프로 전달되고 완성된 프롬프트가 LLM으로 전달되고 LLM의 응답 결과 문자열 파서로 전달되서 문자열만 추출한다.
str_chain = prompt | llm | output_parser
response = str_chain.invoke({'query': '리비안의 설립 년도는 언제인가요?'})

In [20]:
response

'리비안(Rivian)은 2009년에 설립되었습니다. 이 회사는 전기차를 전문으로 하며, 특히 전기 픽업트럭과 SUV 모델로 주목받고 있습니다.'

## JSON 파싱 - JsonOutputParser

LLM의 응답 결과(AIMessage 객체)에는 JSON 형태의 데이터가 포함되므로 이를 딕셔너리나 리스트 구조로 변환해서 사용하면 편리하다.

데이터를 JSON 형식으로 해석하고 변환하기 위해서 JsonOutputParser를 import 한다.

In [21]:
from langchain_core.output_parsers import JsonOutputParser

In [22]:
# JSON 파서 JsonOutputParser의 객체를 만든다.
json_parser = JsonOutputParser()

# chain은 prompt와 llm이 연결된 파이프라인이다.
# 질문에 'JSON 형식'으로 응답해달라고 요청했으므로 모델은 대략 {"회사명": "테슬라", ...}와 같은 문자열이 반환된다.
# 리턴된 값은 JSON을 가장한 문자열 형태이므로 파이썬에서 바로 다루기 어려운 상태이다.
response = chain.invoke({'query': '테슬라 창업자는 누구인가요? JSON 형식으로 응답해주세요. 모든 내용을 한글로 출력해주세요.'})
print(type(response.content))
print(response)
print('-' * 100)

# JSON 형식으로 파싱한다.
json_response = json_parser.invoke(response)
print(type(json_response))
print(json_response)

<class 'str'>
content='```json\n{\n  "창업자": {\n    "이름": "엘론 머스크",\n    "출생연도": 1971,\n    "국적": "미국",\n    "직업": "기업가, 엔지니어, 발명가",\n    "회사": "테슬라"\n  }\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 53, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4f8a068d33', 'id': 'chatcmpl-EKCEv1LIDi0nTed6tlsxv9cnewWiO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a069da-66ab-7830-b0bf-5df5b41d9752-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 73, 'total_tokens': 126, 'input_tok

In [25]:
pprint(json_response['창업자'])

{'국적': '미국', '이름': '엘론 머스크', '직업': '기업가, 엔지니어, 발명가', '출생연도': 1971, '회사': '테슬라'}


# Schema 지정 - PydanticOutputParser

LLM의 응답 결과(AIMessage 객체)로 부터 우리가 정의한 데이터 모델(특정 클래스 구조)에 맞춰 답변을 받아낸다.

모델의 답변을 pydantic 클래스 객체로 변환하기 위해 PydanticOutputParser를 import 한다.

In [26]:
from langchain_core.output_parsers import PydanticOutputParser

LangChain이 업데이트되면서 내부 패키지 구조가 변경되서 langchain_core.pydantic_v1를 더 이상 지원하지 않는다.  
`from langchain_core.pydantic_v1 import BaseModel, Field, validator`  
현재의 LangChain은 langchain_core.pydantic_v1 대신 파이썬의 표준 pydantic을 직접 사용한다.

`BaseModel`: pydantic에서 모델을 정의할 때 상속받는 최상위 클래스이다.  
`Field`: 모델 내부 각 필드의 세부 설정 및 제약 조건을 정의한다.  
`validator`: 기본 타입 검사 외에 사용자가 직접 커스텀 검증 로직을 추가할 때 사용하는 데코레이터이다.

In [27]:
from pydantic import BaseModel, Field, validator

데이터 모델(클래스)을 정의한다.  
인물 정보를 담을 Person이라는 클래스를 pydantic의 BaseModel 클래스를 상속받아 만든다.

In [28]:
# BaseModel 클래스를 상속받으면 이 클래스는 단순한 클래스가 아니라, 데이터 타입 검증과 자동 형변환 기능이 내장된 pydantic 데이터 모델로 동작한다.
class Person(BaseModel):
    # 독스트링(docstring)으로 클래스의 설명이다.
    # AI 프레임워크와 연결될 때 AI 모델에게 이 클래스가 전체적으로 무엇을 의미하는지 안내하는 힌트(prompt)역할을 한다.
    '''사람과 그 사람의 직함 또는 직위에 대한 정보.'''
    # 'name: str'는 name 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    # '...'은 이 값이 필수 입력 항목임을 뜻한다. 데이터가 입력될 때 name 값이 누락되면 에러가 발생된다.
    # description 속성은 필드에 대한 설명 메타 데이터로 AI가 데이터를 분석해서 JSON 데이터로 변환할 때, 어떤 정보를 할당할지 판단하는 가이드라인으로 활용된다.
    name: str = Field(..., description='사람의 이름')
    # 'title: str'는 title 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    title: str = Field(..., description='사람의 직함 또는 직위')

PydanticOutputParser가 자동으로 만들어주는 프롬프트

In [29]:
# 위에서 만든 Person 클래스를 기준으로 작동하는 파서를 만든다.
# 이 파서는 AI의 답변을 감시하며 '이름'과 '직함 또는 직위'가 제대로 들어왔는 확인하고 파이썬 객체로 바꿔줄 준비를 한다.
person_parser = PydanticOutputParser(pydantic_object=Person)

print('PydanticOutputParser 프롬프트')
# get_format_instructions() 메소드는 AI가 답변을 어떻게 JSON 형식으로 구성해야 하는지 설명하는 프롬프트를 자동으로 생성한다.
print(person_parser.get_format_instructions())

PydanticOutputParser 프롬프트
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "사람과 그 사람의 직함 또는 직위에 대한 정보.", "properties": {"name": {"description": "사람의 이름", "title": "Name", "type": "string"}, "title": {"description": "사람의 직함 또는 직위", "title": "Title", "type": "string"}}, "required": ["name", "title"]}
```


앞서 정의한 PydanticOutputParser 모델의 Person 클래스로 정의한 출력 규칙을 AI에게 전달할 최종 프롬프트를 만든다.

In [32]:
# partial() 메소드로 프롬프트에 PydanticOutputParser를 넘겨준다.
# partial(변수 이름, PydanticOutputParser가 자동으로 만들어준 프롬프트)
prompt = ChatPromptTemplate.from_messages([
    ('system', '사용자 질의에 답변해주세요. 출력 결과를 JSON 형태의 태그로 감싸주세요.\n{format_instructions}'),
    ('human', '{query}')
]).partial(format_instructions=person_parser.get_format_instructions())

print('ChatPromptTemplate 프롬프트')
print(prompt.format(query='테슬라 창업자는 누구인가요?'))

ChatPromptTemplate 프롬프트
System: 사용자 질의에 답변해주세요. 출력 결과를 JSON 형태의 태그로 감싸주세요.
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "사람과 그 사람의 직함 또는 직위에 대한 정보.", "properties": {"name": {"description": "사람의 이름", "title": "Name", "type": "string"}, "title": {"description": "사람의 직함 또는 직위", "title": "Title", "type": "string"}}, "required": ["name", "title"]}
```
Human: 테슬라 창업자는 누구인가요?


In [37]:
# prompt: 사용자 질문(query)을 받아, 사전에 정의한 JSON 형식의 지시사항(format_instructions)과 합쳐서 프롬프트를 만든다.
# llm: 프롬프트를 입력받아 AI 답변을 생성한다.
# person_parser: AI가 내놓은 답변을 앞서 만든 Person 클래스(name, title) 구조에 맞는지 검사하고, 문자열을 Person 클래스 객체로 변환한다.
person_chain = prompt | llm | person_parser

# invoke() 메소드로 person_chain을 실행하면 '테슬라 창업자는 누구인가요?'라는 메시지가 prompt의 {query} 변수로 전달된다. 프롬프트의 {format_instructions}
# 변수에는 PydanticOutputParser가 자동으로 만들어준 PydanticOutputParser 프롬프트가 저장되어 있다.
# 최종 완성된 프롬프트의 출력이 LLM으로 전달되고 JSON 형태인 LLM의 출력이 PydanticOutputParser로 전달된다.
# PydanticOutputParser는 Person 클래스에 정의한 스키마 구조에 맞춰서 최종 결과를 출력한다.
response = person_chain.invoke({'query': '테슬라 창업자는 누구인가요?'})

In [38]:
response

Person(name='일론 머스크', title='CEO 및 공동 창립자')

In [39]:
response.name

'일론 머스크'

In [40]:
response.title

'CEO 및 공동 창립자'

# Chat Completion Methods

Chat Completion Methods는 LLM에서 대화형 문맥을 기반으로 응답을 생성하는 API 방식 및 메소드 체계를 의미한다.

LLM이 답변을 생성할 때, 한꺼번에 결과를 내놓는 것이 아니라 생성되는 즉시 실시간으로 화면에 뿌려주는 `스트리밍(Streaming)` 방식을 구현한다. 마치 GPT가 답변을 한 글자씩 타이핑하듯 보여주는 것과 같은 원리이다.

파이썬의 내장 라이브러리인 time은 sleep() 메소드로 인수로 지정한 시간만큼 프로그램을 일시적으로 멈춘다. 프로그램의 실행 속도를 인위적으로 조절하거나 대기 시간을 줄 때 사용한다.

In [41]:
import time

invoke() 메소드로 LLM을 실행하면 완성된 출력을 받아오는데 stream() 메소드로 LLM을 실행하면 응답을 실시간 스트림으로 받아온다.

In [44]:
llm.invoke('테슬라 창업자는 누구인가요?')

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하고 회사의 비전을 이끌어가면서 테슬라의 성장에 중요한 역할을 하게 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 17, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f56c486beb', 'id': 'chatcmpl-EKDEQrxbz8gxZqtKhhlvwMmjN7qRu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06a14-9334-79d0-bc78-05ae0a676d91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17

In [45]:
llm.stream('테슬라 창업자는 누구인가요?')

<generator object BaseChatModel.stream at 0x000002E5045B3870>

In [52]:
# 스트리밍 반복문
# stream() 메소드는 LLM에게 질문을 던지되, 전체 답변이 완성될 때가지 기다리지 않고 부분적인 답변 조각이 생성될 때마다 즉시 내보낸다.
for chunk in llm.stream('테슬라 창업자는 누구인가요?'):
    # chunk.content: 모델이 보낸 조각 데이터에서 실제 텍스트만 추출한다.
    # end='': print() 함수로 출력할 때 줄바꿈을 하지 않는다.
    # flush=True: 출력할 데이터를 버퍼(임시 저장소)에 저장했다가 출력하지 말고 즉시 출력한다. 지연 없이 보여주기 위한 필수 옵션이다.
    print(chunk.content, end='', flush=True)
    time.sleep(0.1)

테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 방향성을 크게 변화시켰습니다. 이후 그는 테슬라의 가장 유명한 얼굴이 되었습니다.

여러 개의 질문을 하나씩 차례대로 묻지 않고, 한꺼번에 묶어서(batch) 모델에게 전달해서 처리한다. 대량의 데이터를 처리할 때 속도와 효율성을 높여준다.

In [53]:
# AI에게 물어볼 여러 개의 질문들을 파이썬 리스트 형태로 생성한다.
questions = [
    '테슬라의 창업자는 누구인가요?',
    '리비안의 창업자는 누구인가요?'
]

# 일괄 처리를 실행한다.
# batch() 메소드는 인수로 질문들이 저장된 리스트를 넘겨서, 리스트에 저장된 모든 질문들을 병렬 혹은 최적화된 방식으로 모델에게 보낸다.
# 질문이 10개라면 invoke() 메소드를 10번 호출하는 것보다 batch() 메소드를 한 번 쓰는 것이 네트워크 지연 시간을 줄여 훨씬 효율적이다.
response = llm.batch(questions)

In [54]:
response

[AIMessage(content='테슬라의 창립자는 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)입니다. 그러나 일론 머스크(Elon Musk)는 2004년에 테슬라에 투자하고 이후 CEO로 취임하면서 회사의 주요 인물로 자리잡았습니다. 일론 머스크는 테슬라의 비전과 방향성을 이끌어가는 데 중요한 역할을 했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 18, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_54ec6000d4', 'id': 'chatcmpl-EKDflWZMABT6Luox4GBEYDWEg7DRx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06a2e-6ff1-7eb1-ad08-6168129ba504-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'outpu

In [59]:
for res in response:
    # print(res.content)
    # pretty_print() 메소드는 LangChain의 메시지 객체가 제공하는 메소드로 단순히 텍스트만 출력하는 것이 아니고 메시지의 역할(System, Human, Ai)을 구분하여
    # 사람이 읽기 편한 가독성 높은 형태로 출랙한다.
    res.pretty_print()
    print()

================================== Ai Message ==================================

테슬라의 창립자는 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)입니다. 그러나 일론 머스크(Elon Musk)는 2004년에 테슬라에 투자하고 이후 CEO로 취임하면서 회사의 주요 인물로 자리잡았습니다. 일론 머스크는 테슬라의 비전과 방향성을 이끌어가는 데 중요한 역할을 했습니다.

================================== Ai Message ==================================

리비안(Rivian)의 창업자는 RJ 스케링(RJ Scaringe)입니다. 그는 2009년에 리비안을 설립하였으며, 전기차 제조업체로서 SUV와 픽업트럭을 주로 생산하고 있습니다. RJ 스케링은 MIT에서 기계공학 박사 학위를 취득한 후 리비안을 창립하게 되었습니다.



# Runnable

Runnable은 LCEL의 핵심 개념으로, RAG 시스템을 구성하는 모든 구성 요소(프롬프트, 모델, 리트리버, 출력 파서 등)를 연결하고 실행하는 인터페이스 표준입니다.

<img src="./runnable.png" width="1200" align="left" />

In [64]:
# 문서 검색기를 설정한다.
# as_retriever() 메소드로 벡터 저장소(vectorstore)를 검색 옵션(search_kwargs)을 지정해서 문서 검색기로 변환한다.
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

query = '테슬라 창업자는 누구인가요?'
# invoke() 메소드로 질문을 넘겨서 문서 검색기에서 문서 검색을 실행한다.
retriever_doc = retriever.invoke(query)
retriever_doc

[Document(id='57d51b60-fd96-42a2-8545-dd439419bedf', metadata={'source': './data\\테슬라_KR.txt'}, page_content='테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.'),
 Document(id='a62a9463-a228-42f5-ad17-77395c84bfbe', metadata={'source': './data\\테슬라_KR.txt'}, page_content='2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.')]

In [68]:
# 검색된 문서의 텍스트 추출 및 결합
retriever_doc_text = '\n'.join([doc.page_content for doc in retriever_doc])
pprint(retriever_doc_text)

('테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 '
 '마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. '
 '머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 '
 '이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n'
 '2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 '
 '테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 '
 '세계 전기차 시장의 약 12.9%를 차지했습니다.')


## RunnableParallel

파이썬 내장 라이브러리인 operator에서 딕셔너리에서 특정 key에 해당되는 value를 추출하는 함수 itemgetter를 사용하기 위해 import 한다.

In [77]:
from operator import itemgetter

In [88]:
sample_dict = {'name': '홍길동', 'age': 30, 'gender': True, 'city': '서울'}
print(sample_dict)
print(sample_dict.get('name'))
print(sample_dict['name'])

get_name = itemgetter('name')
print(get_name)
print(get_name(sample_dict))
print('-' * 100)

users = [
    {'name': '임꺽정', 'score': 85},
    {'name': '장길산', 'score': 95},
    {'name': '일지매', 'score': 70},
]
print(users)

# 'score' 키를 기준으로 오름차순 정렬
sorted(users, key=itemgetter('score'))

{'name': '홍길동', 'age': 30, 'gender': True, 'city': '서울'}
홍길동
홍길동
operator.itemgetter('name')
홍길동
----------------------------------------------------------------------------------------------------
[{'name': '임꺽정', 'score': 85}, {'name': '장길산', 'score': 95}, {'name': '일지매', 'score': 70}]


[{'name': '일지매', 'score': 70},
 {'name': '임꺽정', 'score': 85},
 {'name': '장길산', 'score': 95}]

LangChain에서 입력받은 데이터를 병렬로 동시에 실행하여 새로운 딕셔너리 형태로 가공하기 위해 RunnableParallel을 import 한다.

In [69]:
from langchain_core.runnables import RunnableParallel

In [90]:
# RunnableParallel를 구성한다.
# 어떤 입력이 들어오든 '{'context': ..., 'question': ...}' 형태의 딕셔너리로 변환해 주는 역할을 한다.
runnable = RunnableParallel({
    # itemgetter() 함수는 LangChain에서 사용되면 이전 단계의 결과에서 특정 필드만 뽑아 프롬프트나 다음 Runnable로 전달할 때 사용한다.
    # invoke() 메소드를 실행하면 넘어오는 딕셔너리에서 'context'라는 key에 할당된 value를 가저와서 새 딕셔너리의 'context'라는 key에 value로 할당한다.
    'context': itemgetter('context'),
    'question': itemgetter('question'),
})

# RunnableParallel 객체를 실행한다.
# invoke() 메소드는 인수로 Runnable 객체, 함수, 딕셔너리 중 하나를 넘겨서 RunnableParallel 객체를 실행한다.
response = runnable.invoke({'context': retriever_doc_text, 'question': query})
response

{'context': '테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.',
 'question': '테슬라 창업자는 누구인가요?'}

## RunnablePassthrough

입력 데이터를 변형없이 그대로 다음 단계로 통과(Passthrough) 시키거나, 기존 데이터에 새로운 키를 추가하기 위해 RunnablePassthrough를 import 한다.

In [91]:
from langchain_core.runnables import RunnablePassthrough

In [96]:
# RunnablePassthrough를 구성한다.
runnable = RunnableParallel({
    'context': itemgetter('context'),
    'question': itemgetter('question'),
    # RunnablePassthrough()는 invoke() 메소드가 실행되서 RunnableParallel로 넘어오는 딕셔너리를 그대로 'question22'라는 key에 value로 할당한다.
    'question22': RunnablePassthrough()
})

response = runnable.invoke({'context': retriever_doc_text, 'question': query})
response

{'context': '테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.',
 'question': '테슬라 창업자는 누구인가요?',
 'question22': {'context': '테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.',
  'question': '테슬라 창업

## RunnableLambda

사용자 정의 함수나 lambda 식을 LangChain에서 사용 가능한 객체로 매핑하기 위해서 RunnableLambda를 import 한다.

In [97]:
from langchain_core.runnables import RunnableLambda

In [112]:
# 사용자 정의 함수
# 문자열을 인수로 받아서 공백을 기준으로 나눈 뒤, 그 단어의 개수를 리턴하는 함수
def count_words(text):
    # print(text)
    return len(text['question'].split())

In [113]:
count_words({'context': retriever_doc_text, 'question': query})

3

In [114]:
# RunnableLambda를 구성한다.
runnable = RunnableParallel(
    # invoke() 메소드가 실행되면 RunnableParallel로 넘어오는 딕셔너리를 그대로 'question'이라는 key에 value로 할당한다.
    question = RunnablePassthrough(),
    # invoke() 메소드가 실행되면 RunnableParallel로 넘어오는 딕셔너리를 count_words라는 함수로 던지고 함수의 리턴값을 'count_words'라는 key에 value로 할당한다.
    count_words = RunnableLambda(count_words)
)

response = runnable.invoke({'context': retriever_doc_text, 'question': query})
response

{'question': {'context': '테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.\n2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.',
  'question': '테슬라 창업자는 누구인가요?'},
 'count_words': 3}

#  LCEL을 활용하는 RAG 파이프라인 만들기

## 프롬프트 템플릿

모델이 외부 지식을 배제하고 제공된 정보 내에서만 답변하도록 강제하는 RAG 시스템의 프롬프트를 만든다.

In [118]:
# 프롬프트를 구성하는 문자열을 정의한다.
# 지시사항: 문맥을 바탕으로 답변하라. 외부 지식을 쓰지 마라. 쓸데없이 말 지어내지 말고 모르면 모른다고 해라.
template = '''
다음 문맥(context)을 바탕으로 질문에 답변하세요.
외부 정보나 지식을 전혀 사용하지 마세요.
문맥에 답이 포함되어 있지 않다면 '잘 모르겠습니다.'라고 답변하세요.

[문맥(context)]
{context}

[질문]
{question}

[답변]
'''

# 프롬프트를 구성하는 문자열로 프롬프트 템플릿 객체를 만든다.
prompt = ChatPromptTemplate.from_template(template)
prompt.pretty_print()

================================ Human Message =================================


다음 문맥(context)을 바탕으로 질문에 답변하세요.
외부 정보나 지식을 전혀 사용하지 마세요.
문맥에 답이 포함되어 있지 않다면 '잘 모르겠습니다.'라고 답변하세요.

[문맥(context)]
{context}

[질문]
{question}

[답변]



## 검색기

사용자의 질문과 관련된 문서를 벡터 저장소에서 찾아내고, 찾아낸 문서들을 하나의 텍스트로 합친다.

In [125]:
# 벡터 스토어(vectorstore)에서 as_retriever() 메소드로 질문과 유사한 문서를 가져올 개수를 지정해서 검색기를 설정한다.
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

# 검색기가 검색한 Document 객체에서 실제 텍스트 내용(page_content)을 '\n\n'를 넣어서 하나의 긴 문자열로 합쳐주는 함수를 만든다.
def format_doc(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

# 리트리버 체인을 만든다.
retriever_chain = retriever | format_doc

# invoke() 메소드가 실행되면 인수로 지정된 질문('테슬라 창업자는 누구인가요?')이 검색기에 전달되고 검색기는 질문과 유사한 문서 2개를 리턴한다.
# 검색기가 리턴한 질문과 유사한 문서가 format_doc 함수의 인수로 전달되 문서를 '\n\n'로 연결한 하나의 문자열이 리턴되서 response에 저장된다.
response = retriever_chain.invoke('테슬라 창업자는 누구인가요?')

In [126]:
print(response)

테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다. 2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다. 머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다. 회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다. 테슬라는 2010년 6월 나스닥에 상장되었습니다.

2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37.65% 증가했습니다. 2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다. SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12.9%를 차지했습니다.


## RAG

앞의 모든 컴포넌트(프롬프트, 모델, StrOutputParser)들을 chain으로 엮어서 질문에 대해 문서를 검색하고 답변하는 RAG를 만든다.

In [127]:
# RAG chain을 만든다.
rag_chain = (
    # 프롬프트의 변수({context}, {question})에 입력으로 넣어줄 내용을 정의하는 단계
    # {context}에는 사용자 질문을 검색기에 던져서 유사한 문서를 얻어와서 한 문장으로 합치는 retriever_chain의 실행 결과를 넣어줘야 한다.
    # {question}에는 rag_chain 체인을 실행할 때 invoke() 메소드의 인수인 질문 '테슬라 창업자는 누구인가요?'를 넣어줘야 한다.
    # 'context'에는 retriever_chain를 할당하고 'question'에는 질문이 그대로 통과되게 RunnablePassthrough() 객체를 할당한다.
    {'context': retriever_chain, 'question': RunnablePassthrough()}
    # 프롬프트를 준비된 'context'와 'question'을 이용해서 변수에 내용을 채워 완성하는 단계
    | prompt
    # 완성된 프롬프트를 LLM 모델에 전달하여 답변을 생성하는 단계
    | llm
    # 모델이 응답한 답변에서 텍스트만 얻어내는 단계
    | StrOutputParser()
)

# invoke()를 실행하면 관련 문서 검색 => 프롬프트 조립 => 답변 생성 => 텍스트 추출이 순차적으로 실행된다.
# invoke() 메소드가 실행되면 retriever_chain이 실행된 결과인 질문을 벡터 스토어에서 검색한 결과를 한 문장으로 연결한 문자열이 딕셔너리의 'context' key에
# 할당되고 invoke() 메소드의 인수로 지정된 질문 '테슬라 창업자는 누구인가요?'가 'question' key에 RunnablePassthrough()에 의해서 '테슬라 창업자는 누구인가요?'
# 그대로 할당되서 {'context': '테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를...', 'question': '테슬라 창업자는 누구인가요?'} 딕셔너리가 완성된다.
# 완성된 딕셔너리가 프롬프트로 전달되서 {context}, {question} 변수의 내용을 채워서 프롬프트가 완성된다.
# 완성된 프롬프트가 LLM으로 전달되서 프롬프트에 정의한 규칙대로 질문을 하고 최종 답변을 얻어온다.
# 얻어온 최종 답변이 StrOutputParser() 객체에 전달되서 텍스트만 얻어온다.
response = rag_chain.invoke('테슬라 창업자는 누구인가요?')

In [128]:
response

'테슬라의 창업자는 마틴 에버하드와 마크 타페닝입니다.'

# Gradio 챗봇

앞서 완성한 rag_chain을 사용해서 챗봇 웹 앱을 만든다.

In [129]:
import gradio as gr

In [132]:
# 채팅 처리 함수
# message: 질문
# history: 이전 대화 내용, rag_chain에 기억 기능이 없으므로 사용하지 않고 비워둔다.
def answer_invoke(message, history):
    response = rag_chain.invoke(message)
    return response

# Gradio 인터페이스 생성
chatbot = gr.ChatInterface(fn=answer_invoke, title='QA Bot')

# 실행
chatbot.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [133]:
chatbot.close()

Closing server running on port: 7862


# Gradio 챗봇 - stream 실행

답변을 글자가 한 글자씩 써지는 스트리밍 방식으로 챗봇을 구현한다.

In [134]:
# 스트리밍 방식으로 처리하는 함수를 구현한다.
def answer_stream(message, history):
    # 모델이 생성하는 글자 조각들을 하나로 합쳐서 저장할 빈 변수를 선언한다.
    partial_message = ''
    # 스트리밍 루프
    # invoke() 메소드 대신 stream() 메소드를 사용해서 답변이 다 완성될 때까지 기다리지 않고, 생성되는 즉시 내보낸다.
    for chunk in rag_chain.stream(message):
        # partial_message 변수에 답변 누적과 지연 시간을 설정한다.
        if chunk is not None:
            # 새로 들어온 글자 조각을 기존 문자열 뒤에 이어붙인다.
            partial_message += chunk
            # 지연 시간을 설정한다.
            time.sleep(0.1)
            # 제네레이너 반환
            # yield는 함수를 일반 함수가 아닌 제네레이터로 만든다. return은 값을 돌려주고 함수를 종료하지만, yield는 값을 중간중간 계속 전달하면서
            # 함수가 완전히 종료되거나 return을 만날 때까지 유지한다.
            # Gradio는 이 yield 값을 실시간으로 받아서 채팅창의 글자를 업데이트 한다.
            yield partial_message
    # ===== for

chatbot = gr.ChatInterface(fn=answer_stream, title='QA Bot')
chatbot.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [ ]:
chatbot.close()